In [ ]:
# make browse images for BDE images and gifs vs temp (match id to temp table) 

In [ ]:
folder = Path("/home/bekah/m3_papers/20260311/bde_fits")
temps = pd.read_fwf("/home/bekah/m3_papers/20260310/m3_detector_temperature.tab")
temp = df.loc[df["id"] == obsid, "temp"].iloc[0]


In [ ]:
from pathlib import Path
import numpy as np
from astropy.io import fits
from PIL import Image
import pandas as pd 
folder = Path("/home/bekah/m3_papers/20260311_bde_darkcurrent/bde_fits")

# temp df 
#temps = pd.read_fwf("/home/bekah/m3_papers/20260310/m3_detector_temperature.tab")


flat_dir = folder / "flat_fits"
browse_dir = folder / "browse"
flat_dir.mkdir(exist_ok=True)
browse_dir.mkdir(exist_ok=True)
for fits_file in folder.glob("*.fits"):
    
    #temp = df.loc[df["id"] == obsid, "temp"].iloc[0]
    
    with fits.open(fits_file) as hdul:
        data = hdul[0].data
        
        lo = 0 
        hi = 5
        stretched = np.clip((data - lo) / (hi - lo), 0, 1)
    
        browse_uint8 = (stretched * 255).astype(np.uint8)
    
        img = Image.fromarray(browse_uint8, mode="L")
    
        out_jpg = browse_dir / f"{fits_file.stem}_browse.jpg"
        img.save(out_jpg, "JPEG", quality=95)


In [ ]:
from pathlib import Path
import pandas as pd

folder = Path("/home/bekah/m3_papers/20260311/bde_fits")

temps = pd.read_fwf("/home/bekah/m3_papers/20260310/m3_detector_temperature.tab")

temps = temps.set_index("id")

records = []

for fits_file in folder.glob("*.fits"):
    
    obsid = fits_file.stem.split("_")[0].upper()
    
    # no target
    if "T" in obsid[0:3]:
        continue
    
    if obsid in temps.index:
        temp = temps.loc[obsid, "temp"]
        records.append((fits_file, temp))

records_sorted = sorted(records, key=lambda x: x[1])

fits_sorted = [r[0] for r in records_sorted]

In [ ]:
fits_sorted

In [ ]:
from PIL import ImageDraw, ImageFont
from astropy.io import fits
import numpy as np
from PIL import Image

#fits_file = "/home/bekah/m3_papers/20260311_bde_darkcurrent/m3g20081222t223435_l1b_rdn.fits"
fits_file = "/home/bekah/m3_papers/20260311_bde_darkcurrent/m3g20090626t142653_l1b_rdn.fits"
with fits.open(fits_file) as hdul:
    data = hdul[0].data

    bands, rows, cols = data.shape
    tmp = data.transpose(1, 0, 2)  
    image = tmp

    out_fits = "lb1_weirdbde.fits"
    fits.writeto(out_fits, image, overwrite=True)
